In [31]:
!pip install fastdtw pandas numpy scipy scikit-learn


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [32]:
import pandas as pd
import numpy as np
from fastdtw import fastdtw
from scipy.spatial.distance import euclidean
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import warnings

warnings.filterwarnings('ignore')

**1. TIỀN XỬ LÝ & TRÍCH XUẤT ĐẶC TRƯNG GỘP FRAME**

In [33]:
def min_max_scale(series):
    return (series - series.min()) / (series.max() - series.min() + 1e-9)

def preprocess_data(df):
    df = df.sort_values(by=['Joint_Name', 'Frame']).reset_index(drop=True)
    mask = df['Visibility'] < 0.5
    df.loc[mask, ['X', 'Y', 'Z']] = np.nan
    df[['X', 'Y', 'Z']] = df.groupby('Joint_Name')[['X', 'Y', 'Z']].transform(
        lambda x: x.interpolate(method='linear', limit_direction='both')
    )
    df.fillna(0, inplace=True) 
    df['X_norm'] = min_max_scale(df['X'])
    df['Y_norm'] = min_max_scale(df['Y'])
    df['Z_norm'] = min_max_scale(df['Z'])
    return df.sort_values(by=['Frame', 'Joint_Name']).reset_index(drop=True)

def build_frame_level_dataset(df_student, df_teacher):
    def get_vectors(df):
        df_unique = df.drop_duplicates(subset=['Frame', 'Joint_Name'])
        wide = df_unique.pivot(index='Frame', columns='Joint_Name', values=['X_norm', 'Y_norm', 'Z_norm'])
        wide.fillna(0, inplace=True)
        return wide.values, wide.index.tolist()

    vec_s, frames_s = get_vectors(df_student)
    vec_t, frames_t = get_vectors(df_teacher)
    distance, path = fastdtw(vec_s, vec_t, dist=euclidean)
    
    aligned_pairs = [{'Frame_s': frames_s[s_idx], 'Frame_t': frames_t[t_idx]} for s_idx, t_idx in path]
    df_path = pd.DataFrame(aligned_pairs)
    
    df_s_aligned = pd.merge(df_path, df_student, left_on='Frame_s', right_on='Frame').drop(columns=['Frame'])
    df_t_aligned = pd.merge(df_path, df_teacher, left_on='Frame_t', right_on='Frame').drop(columns=['Frame'])
    
    df_s_aligned = df_s_aligned.rename(columns={'X_norm': 'X_s', 'Y_norm': 'Y_s', 'Z_norm': 'Z_s', 'Label': 'Label_s'})
    df_t_aligned = df_t_aligned.rename(columns={'X_norm': 'X_t', 'Y_norm': 'Y_t', 'Z_norm': 'Z_t'})
    
    # Xử lý Label (Đề phòng trường hợp Label_s_s do merge cũ)
    label_col = 'Label_s' if 'Label_s' in df_s_aligned.columns else 'Label'
    
    df_merged = pd.merge(
        df_s_aligned[['Frame_s', 'Frame_t', 'Joint_Name', 'X_s', 'Y_s', 'Z_s', label_col]], 
        df_t_aligned[['Frame_s', 'Frame_t', 'Joint_Name', 'X_t', 'Y_t', 'Z_t']], 
        on=['Frame_s', 'Frame_t', 'Joint_Name']
    )
    
    # 1. Tính Error & Cosine cho từng khớp
    df_merged['Joint_Error'] = np.sqrt((df_merged['X_s'] - df_merged['X_t'])**2 + (df_merged['Y_s'] - df_merged['Y_t'])**2 + (df_merged['Z_s'] - df_merged['Z_t'])**2)
    dot = (df_merged['X_s'] * df_merged['X_t']) + (df_merged['Y_s'] * df_merged['Y_t']) + (df_merged['Z_s'] * df_merged['Z_t'])
    norm_s = np.sqrt(df_merged['X_s']**2 + df_merged['Y_s']**2 + df_merged['Z_s']**2)
    norm_t = np.sqrt(df_merged['X_t']**2 + df_merged['Y_t']**2 + df_merged['Z_t']**2)
    df_merged['Joint_Cosine'] = dot / (norm_s * norm_t + 1e-9)
    
    # 2. Xoay ngang dữ liệu (Pivot) - gom 13 khớp vào 1 dòng (1 Frame)
    wide_error = df_merged.pivot_table(index='Frame_s', columns='Joint_Name', values='Joint_Error', fill_value=0)
    wide_error.columns = [f"Error_{c}" for c in wide_error.columns]
    
    wide_cosine = df_merged.pivot_table(index='Frame_s', columns='Joint_Name', values='Joint_Cosine', fill_value=0)
    wide_cosine.columns = [f"Cosine_{c}" for c in wide_cosine.columns]
    
    wide_label = df_merged.pivot_table(index='Frame_s', columns='Joint_Name', values=label_col, fill_value=0)
    wide_label.columns = [f"Label_{c}" for c in wide_label.columns]
    
    frame_dataset = pd.concat([wide_error, wide_cosine, wide_label], axis=1).reset_index()
    
    # 3. TẠO LABEL THEO FRAME
    # Yêu cầu: Frame Pass khi có >= 7 khớp Pass (Quá nửa trong số 13 khớp)
    label_cols = [c for c in frame_dataset.columns if c.startswith('Label_')]
    frame_dataset['Total_Correct_Joints'] = frame_dataset[label_cols].sum(axis=1)
    frame_dataset['Frame_Label'] = (frame_dataset['Total_Correct_Joints'] >= 7).astype(int)
    
    # Xóa các cột Label của từng khớp vì không cần dùng cho ML nữa
    frame_dataset.drop(columns=label_cols + ['Total_Correct_Joints'], inplace=True)
    
    return frame_dataset

**2. NẠP DỮ LIỆU & TẠO TẬP TRAIN/TEST**

In [34]:
dance_pairs = [
    {"id": 1, "student": "student_dance_1.csv", "teacher": "teacher_dance_1.csv"},
    {"id": 2, "student": "student_dance_2.csv", "teacher": "teacher_dance_2.csv"},
    {"id": 3, "student": "student_dance_3.csv", "teacher": "teacher_dance_3.csv"},
    {"id": 4, "student": "student_dance_4.csv", "teacher": "teacher_dance_4.csv"},
    {"id": 5, "student": "student_dance_5.csv", "teacher": "teacher_dance_5.csv"},
    {"id": 6, "student": "student_dance_6.csv", "teacher": "teacher_dance_6.csv"}, 
    {"id": 7, "student": "student_dance_7.csv", "teacher": "teacher_dance_7.csv"},
    {"id": 8, "student": "student_dance_8.csv", "teacher": "teacher_dance_8.csv"},
    {"id": 9, "student": "student_dance_9.csv", "teacher": "teacher_dance_9.csv"},
    {"id": 10, "student": "student_dance_10.csv", "teacher": "teacher_dance_10.csv"}
]

all_frames = []

print("=== BẮT ĐẦU TRÍCH XUẤT DỮ LIỆU CẤP ĐỘ FRAME ===")
for pair in dance_pairs:
    try:
        df_s = pd.read_csv(pair["student"])
        df_t = pd.read_csv(pair["teacher"])
        
        df_s_clean, df_t_clean = preprocess_data(df_s), preprocess_data(df_t)
        
        frame_df = build_frame_level_dataset(df_s_clean, df_t_clean)
        frame_df['Video_ID'] = pair['id']
        
        all_frames.append(frame_df)
        print(f"[+] Trích xuất thành công Video {pair['id']}: {len(frame_df)} frames")
    except Exception as e:
        print(f"[-] Lỗi tại Video {pair['id']}: {e}")

master_dataset = pd.concat(all_frames, ignore_index=True)
print(f"\n-> Tổng số Frame thu thập được: {len(master_dataset)}")

=== BẮT ĐẦU TRÍCH XUẤT DỮ LIỆU CẤP ĐỘ FRAME ===
[+] Trích xuất thành công Video 1: 486 frames
[+] Trích xuất thành công Video 2: 368 frames
[+] Trích xuất thành công Video 3: 446 frames
[+] Trích xuất thành công Video 4: 726 frames
[+] Trích xuất thành công Video 5: 338 frames
[+] Trích xuất thành công Video 6: 338 frames
[+] Trích xuất thành công Video 7: 446 frames
[+] Trích xuất thành công Video 8: 483 frames
[+] Trích xuất thành công Video 9: 275 frames
[+] Trích xuất thành công Video 10: 1041 frames

-> Tổng số Frame thu thập được: 4947


**3. HUẤN LUYỆN RANDOM FOREST & ĐÁNH GIÁ**

In [35]:
# 1. CHỌN VIDEO TEST
TEST_VIDEO_IDS = [5,6]  # Cậu tự quyết định Test video nào ở đây

train_data = master_dataset[~master_dataset['Video_ID'].isin(TEST_VIDEO_IDS)]
test_data = master_dataset[master_dataset['Video_ID'].isin(TEST_VIDEO_IDS)]

# 2. TÁCH FEATURES (X) VÀ LABELS (y)
# Bỏ các cột không phải đặc trưng (Frame_s, Video_ID, Frame_Label)
feature_cols = [col for col in master_dataset.columns if col.startswith('Error_') or col.startswith('Cosine_')]

X_train = train_data[feature_cols]
y_train = train_data['Frame_Label']

X_test = test_data[feature_cols]
y_test = test_data['Frame_Label']

print(f"Dữ liệu Train: {X_train.shape[0]} frames")
print(f"Dữ liệu Test : {X_test.shape[0]} frames")

# 3. HUẤN LUYỆN RANDOM FOREST
print("\nĐang huấn luyện mô hình Random Forest...")
rf_model = RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42)
rf_model.fit(X_train, y_train)

# 4. ĐÁNH GIÁ
y_pred = rf_model.predict(X_test)

print("\n======================================================")
print(f"   KẾT QUẢ ĐÁNH GIÁ CẤP ĐỘ FRAME (TEST VIDEO {TEST_VIDEO_IDS})")
print("======================================================\n")

print(classification_report(y_test, y_pred, target_names=['Fail (0)', 'Pass (1)']))

acc = accuracy_score(y_test, y_pred)
print(f"-> ĐỘ CHÍNH XÁC TỔNG THỂ (Accuracy): {acc*100:.2f}%")

# Thống kê chi tiết
cm = confusion_matrix(y_test, y_pred)
print(f"\nCHI TIẾT TRÊN TỔNG SỐ {len(y_test)} FRAMES TEST:")
if cm.shape == (2,2):
    print(f"- Số Frames thực tế FAIL, máy chấm chuẩn FAIL : {cm[0][0]}")
    print(f"- Số Frames thực tế FAIL, máy chấm lọt PASS   : {cm[0][1]} (Báo động giả)")
    print(f"- Số Frames thực tế PASS, máy chấm oan FAIL   : {cm[1][0]} (Đánh hỏng oan)")
    print(f"- Số Frames thực tế PASS, máy chấm chuẩn PASS : {cm[1][1]}")
else:
    print("- Dữ liệu Test chỉ có 1 nhãn (hoàn toàn Pass hoặc hoàn toàn Fail).")
    
# Hiển thị Top 10 Đặc trưng quyết định khung hình đúng/sai
importances = rf_model.feature_importances_
feature_imp_df = pd.DataFrame({'Đặc trưng': feature_cols, 'Mức độ đóng góp (%)': importances * 100})
print("\n=== TOP 10 ĐẶC TRƯNG QUAN TRỌNG NHẤT KHUNG HÌNH ===")
display(feature_imp_df.sort_values(by='Mức độ đóng góp (%)', ascending=False).head(10))

Dữ liệu Train: 4271 frames
Dữ liệu Test : 676 frames

Đang huấn luyện mô hình Random Forest...

   KẾT QUẢ ĐÁNH GIÁ CẤP ĐỘ FRAME (TEST VIDEO [5, 6])

              precision    recall  f1-score   support

    Fail (0)       0.91      0.82      0.86       352
    Pass (1)       0.82      0.92      0.87       324

    accuracy                           0.87       676
   macro avg       0.87      0.87      0.87       676
weighted avg       0.87      0.87      0.87       676

-> ĐỘ CHÍNH XÁC TỔNG THỂ (Accuracy): 86.54%

CHI TIẾT TRÊN TỔNG SỐ 676 FRAMES TEST:
- Số Frames thực tế FAIL, máy chấm chuẩn FAIL : 288
- Số Frames thực tế FAIL, máy chấm lọt PASS   : 64 (Báo động giả)
- Số Frames thực tế PASS, máy chấm oan FAIL   : 27 (Đánh hỏng oan)
- Số Frames thực tế PASS, máy chấm chuẩn PASS : 297

=== TOP 10 ĐẶC TRƯNG QUAN TRỌNG NHẤT KHUNG HÌNH ===


,Đặc trưng,Mức độ đóng góp (%)
19,Cosine_nose,14.977916
6,Error_nose,10.956315
5,Error_left_wrist,8.770232
17,Cosine_left_shoulder,8.299568
1,Error_left_elbow,6.622419
12,Error_right_wrist,3.685739
4,Error_left_shoulder,3.458338
18,Cosine_left_wrist,3.451394
9,Error_right_hip,3.327671
14,Cosine_left_elbow,3.254210
